# Daft.ie New Homes - URL Retrieval
**Step 1 of 2** — Collects all new home listing URLs from daft.ie and saves them to a CSV.

Run each cell from top to bottom. The browser window will open automatically.
You do not need to touch the browser — cookies and popups are handled automatically.

In [ ]:
# ===========================================================================
# CELL 1 - IMPORTS
# ===========================================================================

import re
import time
import random
import tempfile
import math
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

print("Imports loaded.")

In [ ]:
# ===========================================================================
# CELL 2 - OUTPUT FOLDER
# ===========================================================================
# Prompts you to enter where the CSV should be saved.
# Press Enter to use the default folder shown in brackets.
# ===========================================================================

DEFAULT_OUTPUT = r"C:\Users\ciara\myenv\New Homes Ireland\Daft - Dan\Daft-MyHome-Scraper-main\data"

user_input = input(f"Output folder [{DEFAULT_OUTPUT}]: ").strip()
OUTPUT_DIR = Path(user_input) if user_input else Path(DEFAULT_OUTPUT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output folder: {OUTPUT_DIR}")

In [ ]:
# ===========================================================================
# CELL 3 - HELPER FUNCTIONS
# ===========================================================================

BASE_URL = "https://www.daft.ie/new-homes-for-sale/ireland?sort=publishDateDesc&page={page}"


def auto_accept_cookies(driver):
    """
    Automatically clicks the cookie accept button if it appears.
    Daft.ie uses a button with the text 'Accept' or 'Accept All'.
    We try multiple selectors to handle layout changes.
    Does nothing if no cookie banner is found.
    """
    cookie_selectors = [
        # Text-based matches
        "//button[contains(translate(text(),'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ'),'ACCEPT ALL')]",
        "//button[contains(translate(text(),'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ'),'ACCEPT')]",
        # ID/class based
        "//button[@id='onetrust-accept-btn-handler']",
        "//button[contains(@class,'accept')][@data-testid]",
        "//button[contains(@class,'cookie')][@type='button']",
    ]
    for xpath in cookie_selectors:
        try:
            btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            btn.click()
            print("  Cookie banner accepted automatically.")
            time.sleep(1.5)
            return
        except Exception:
            continue
    print("  No cookie banner found - continuing.")


def auto_close_survey(driver):
    """
    Automatically closes survey or feedback popups if they appear.
    These typically have a close button with an X or 'No thanks' text.
    Does nothing if no popup is found.
    """
    survey_selectors = [
        "//button[contains(translate(text(),'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ'),'NO THANKS')]",
        "//button[contains(translate(text(),'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ'),'NO, THANKS')]",
        "//button[contains(translate(text(),'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ'),'CLOSE')]",
        "//button[@aria-label='Close']",
        "//button[@aria-label='close']",
        "//button[contains(@class,'close')]",
        "//*[@id='fsrDeclineButton']",  # ForeSee survey
        "//*[contains(@class,'survey')]//button",
    ]
    for xpath in survey_selectors:
        try:
            btn = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            btn.click()
            print("  Survey popup closed automatically.")
            time.sleep(1)
            return
        except Exception:
            continue


def get_total_pages(driver) -> int:
    """
    Reads the total number of results from the page and calculates
    how many pages to scrape.

    Daft.ie shows text like '1,245 New Homes for Sale' near the top
    of the results page. We extract that number and divide by 20
    (results per page) rounding up.

    Falls back to 20 pages if the count cannot be found.
    """
    RESULTS_PER_PAGE = 20
    FALLBACK_PAGES   = 20

    try:
        soup = BeautifulSoup(driver.page_source, "lxml")

        # Try data-testid first
        count_el = soup.select_one('[data-testid="results-count"]')
        if not count_el:
            # Search all text for a pattern like '1,245 New Homes'
            text = soup.get_text(" ", strip=True)
            m    = re.search(r"([\d,]+)\s+New Home", text, re.IGNORECASE)
            if m:
                total = int(m.group(1).replace(",", ""))
                pages = math.ceil(total / RESULTS_PER_PAGE)
                print(f"  Found {total} total listings -> {pages} pages")
                return pages
        else:
            m = re.search(r"([\d,]+)", count_el.get_text())
            if m:
                total = int(m.group(1).replace(",", ""))
                pages = math.ceil(total / RESULTS_PER_PAGE)
                print(f"  Found {total} total listings -> {pages} pages")
                return pages
    except Exception as e:
        print(f"  Could not read total results ({e}) - using {FALLBACK_PAGES} pages")

    print(f"  Could not find total results - using {FALLBACK_PAGES} pages")
    return FALLBACK_PAGES


def extract_links(html: str) -> set:
    """
    Extracts all new home listing URLs from a results page.
    Targets href values containing /new-home-for-sale/.
    Strips query strings and fragments for clean URLs.
    """
    soup  = BeautifulSoup(html, "lxml")
    links = set()
    for a in soup.select('a[href*="/new-home-for-sale/"]'):
        href = a.get("href")
        if not href:
            continue
        full = urljoin("https://www.daft.ie", href)
        u    = urlparse(full)
        links.add(f"{u.scheme}://{u.netloc}{u.path}")
    return links


print("Helper functions loaded.")

In [ ]:
# ===========================================================================
# CELL 4 - LAUNCH BROWSER AND COLLECT URLs
# ===========================================================================
# Opens Chrome, handles cookies and popups automatically, detects the
# total number of pages, then scrapes every page for listing URLs.
# ===========================================================================

options = Options()
options.add_argument("--window-size=1400,900")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--remote-debugging-port=0")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# Unique temp profile each run to avoid DevToolsActivePort lock errors
profile_dir = tempfile.mkdtemp(prefix="selenium-daft-")
options.add_argument(f"--user-data-dir={profile_dir}")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

all_urls = set()

try:
    # --- Load page 1 ---
    print("Opening daft.ie...")
    driver.get(BASE_URL.format(page=1))
    time.sleep(4)

    # Handle cookie banner automatically
    auto_accept_cookies(driver)

    # Handle survey popup automatically
    auto_close_survey(driver)

    # Scroll to trigger lazy-loaded content then read total pages
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(random.uniform(5.0, 7.0))

    # Detect total pages dynamically
    print("Detecting total pages...")
    total_pages = get_total_pages(driver)

    # Collect links from page 1
    links = extract_links(driver.page_source)
    all_urls.update(links)
    print(f"Page  1/{total_pages} -> {len(links)} links (total so far: {len(all_urls)})")

    # --- Pages 2 onwards ---
    for page in range(2, total_pages + 1):
        try:
            driver.get(BASE_URL.format(page=page))
            time.sleep(random.uniform(1.5, 2.5))

            # Close any popup that appears mid-scrape
            auto_close_survey(driver)

            # Scroll to load lazy content
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(random.uniform(5.0, 9.0))

            links = extract_links(driver.page_source)
            all_urls.update(links)
            print(f"Page {page:2d}/{total_pages} -> {len(links)} links (total so far: {len(all_urls)})")

            time.sleep(random.uniform(1.0, 2.0))

        except Exception as e:
            print(f"Page {page:2d}/{total_pages} -> ERROR: {e}")

finally:
    driver.quit()

print(f"\nTotal unique URLs collected: {len(all_urls)}")

In [ ]:
# ===========================================================================
# CELL 5 - SAVE TO CSV
# ===========================================================================

ts          = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = OUTPUT_DIR / f"daft_listings_newhomes_{ts}.csv"

df = pd.DataFrame(sorted(all_urls), columns=["url"])
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} URLs to:")
print(f"  {output_file}")
print(f"\nSample URLs:")
for url in df["url"].head(5):
    print(f"  {url}")

In [ ]:
# ===========================================================================
# CELL 6 - FIND LATEST CSV (for use in Step 2 scraper)
# ===========================================================================
# Run this any time you want to find the most recently saved URL file
# in your output folder. Copy the path printed below into Step 2.
# ===========================================================================

files = list(OUTPUT_DIR.glob("daft_listings_newhomes_*.csv"))

if files:
    latest = max(
        files,
        key=lambda f: f.stem.split("_")[-2] + f.stem.split("_")[-1]
    )
    print(f"Latest URL file:")
    print(f"  {latest}")
    print(f"  ({pd.read_csv(latest).shape[0]} URLs)")
else:
    print("No URL files found in output folder.")